# MML Chapter 3: Analytic Geometry

> **Premise:** Once you have a vector space, you add structure to measure lengths, angles, and distances — and use that structure to project, orthogonalize, and rotate. This geometry powers PCA, linear regression, and SVMs.

**Source:** Deisenroth, Faisal & Ong — *Mathematics for Machine Learning*, Ch 3 (pp 71–95)

---

## Contents
1. L1 vs L2 norms and their unit balls
2. General inner products via SPD matrices
3. Cauchy-Schwarz inequality: numerical verification
4. Orthogonal projection onto a 1D subspace
5. Gram-Schmidt orthogonalization from scratch
6. 2D rotation matrix: visualizing $R(\theta)$ at multiple angles
7. Exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 100
rng = np.random.default_rng(42)

---
## 1. L1 vs L2 Norm: Unit Balls Side by Side

A **norm** assigns a non-negative length $\|\mathbf{x}\|$ to every vector, satisfying:
- **Absolute homogeneity:** $\|\lambda \mathbf{x}\| = |\lambda| \, \|\mathbf{x}\|$
- **Triangle inequality:** $\|\mathbf{x} + \mathbf{y}\| \leq \|\mathbf{x}\| + \|\mathbf{y}\|$
- **Positive definiteness:** $\|\mathbf{x}\| \geq 0$, with $\|\mathbf{x}\| = 0 \iff \mathbf{x} = \mathbf{0}$

Two common norms on $\mathbb{R}^n$:

$$\|\mathbf{x}\|_1 = \sum_{i=1}^n |x_i| \qquad \text{(Manhattan / L1)}$$
$$\|\mathbf{x}\|_2 = \sqrt{\mathbf{x}^\top \mathbf{x}} = \sqrt{\sum_{i=1}^n x_i^2} \qquad \text{(Euclidean / L2)}$$

The **unit ball** $\{\mathbf{x} : \|\mathbf{x}\| = 1\}$ has characteristic shapes:
- L1 unit ball → diamond (rotated square)
- L2 unit ball → circle

In [ ]:
# Generate points on unit balls for L1 and L2 in 2D
theta = np.linspace(0, 2 * np.pi, 1000)

# L2 unit ball: cos(t), sin(t)
circle_x = np.cos(theta)
circle_y = np.sin(theta)

# L1 unit ball: |x| + |y| = 1  →  parametrize the four segments
# Top-right: (t, 1-t), t ∈ [0,1]; rotate through 4 quadrants
t = np.linspace(0, 1, 250)
l1_x = np.concatenate([ t,  1-t, -t,  -(1-t)])
l1_y = np.concatenate([ 1-t, -t,  -(1-t), t])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# --- L1 ---
ax = axes[0]
ax.fill(l1_x, l1_y, alpha=0.25, color='steelblue')
ax.plot(l1_x, l1_y, 'steelblue', linewidth=2)
ax.set_title("L1 Unit Ball\n$\\{\\mathbf{x} : \\|\\mathbf{x}\\|_1 = 1\\}$", fontsize=12)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.grid(True, alpha=0.3)
# Annotate vertices
for pt, label in [((1,0), '(1,0)'), ((0,1), '(0,1)'), ((-1,0), '(-1,0)'), ((0,-1), '(0,-1)')]:
    ax.annotate(label, pt, textcoords='offset points', xytext=(5,5), fontsize=8)
    ax.plot(*pt, 'ko', markersize=5)

# --- L2 ---
ax = axes[1]
ax.fill(circle_x, circle_y, alpha=0.25, color='coral')
ax.plot(circle_x, circle_y, 'coral', linewidth=2)
ax.set_title("L2 Unit Ball\n$\\{\\mathbf{x} : \\|\\mathbf{x}\\|_2 = 1\\}$", fontsize=12)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.grid(True, alpha=0.3)

# Compute norms for a test vector
v = np.array([0.6, 0.8])
for ax_ in axes:
    ax_.plot(*v, 'k*', markersize=10, label=f'v={v}')

print(f"Test vector v = {v}")
print(f"  ‖v‖₁ = {np.sum(np.abs(v)):.4f}  (= |0.6| + |0.8|)")
print(f"  ‖v‖₂ = {np.linalg.norm(v):.4f}  (= √(0.6² + 0.8²))")

fig.suptitle("Unit Balls: Diamond (L1) vs Circle (L2)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. General Inner Product $\langle \mathbf{x}, \mathbf{y} \rangle = \mathbf{x}^\top A \mathbf{y}$ with an SPD Matrix

The dot product $\mathbf{x}^\top \mathbf{y}$ is just one inner product. Any **symmetric positive definite (SPD)** matrix $A$ defines a valid inner product:
$$\langle \mathbf{x}, \mathbf{y} \rangle_A = \mathbf{x}^\top A \mathbf{y}$$

$A$ is SPD if:
1. $A = A^\top$ (symmetric)
2. $\mathbf{x}^\top A \mathbf{x} > 0$ for all $\mathbf{x} \neq \mathbf{0}$ (positive definite)

Equivalently, all eigenvalues of $A$ are strictly positive.

From the notes:
- $A_1 = [[9,6],[6,5]]$ is SPD — verify that $\mathbf{x}^\top A_1 \mathbf{x} = (3x_1 + 2x_2)^2 + x_2^2 > 0$ always
- $A_2 = [[9,6],[6,3]]$ is **not** SPD — can be negative for some $\mathbf{x}$

In [ ]:
def is_spd(M, tol=1e-9):
    """Check if M is symmetric positive definite."""
    if not np.allclose(M, M.T):
        return False, "Not symmetric", None
    eigenvalues = np.linalg.eigvalsh(M)  # eigenvalues of symmetric matrix
    if np.all(eigenvalues > tol):
        return True, "SPD", eigenvalues
    else:
        return False, f"Non-positive eigenvalue: min={eigenvalues.min():.4f}", eigenvalues

A1 = np.array([[9.0, 6.0],
               [6.0, 5.0]])
A2 = np.array([[9.0, 6.0],
               [6.0, 3.0]])

for name, A in [("A1 (from MML notes)", A1), ("A2 (from MML notes)", A2)]:
    spd, reason, eigs = is_spd(A)
    print(f"{name}:")
    print(f"  Matrix:\n  {A}")
    print(f"  SPD: {spd} — {reason}")
    if eigs is not None:
        print(f"  Eigenvalues: {eigs}")
    print()

# --- Compute generalized inner product for A1 ---
x = np.array([1.0, 2.0])
y = np.array([3.0, -1.0])

ip_standard = x @ y                   # standard dot product
ip_A1       = x @ A1 @ y             # generalized inner product

print(f"x = {x},  y = {y}")
print(f"Standard inner product ⟨x,y⟩ = x·y          = {ip_standard:.4f}")
print(f"Generalized inner product ⟨x,y⟩_A1 = x^T A1 y = {ip_A1:.4f}")

# Verify ⟨x,x⟩_A1 > 0 for many random vectors
test_vecs = rng.standard_normal((1000, 2))
quad_forms_A1 = np.einsum('ij,jk,ik->i', test_vecs, A1, test_vecs)
quad_forms_A2 = np.einsum('ij,jk,ik->i', test_vecs, A2, test_vecs)

print(f"\n1000 random vectors — xᵀ A x:")
print(f"  A1: min={quad_forms_A1.min():.4f}, all positive? {(quad_forms_A1 > 0).all()}")
print(f"  A2: min={quad_forms_A2.min():.4f}, all positive? {(quad_forms_A2 > 0).all()}")

# Visualize the unit balls under each inner product (level sets of xᵀAx = 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
grid = np.linspace(-1.5, 1.5, 400)
X, Y = np.meshgrid(grid, grid)
pts = np.stack([X.ravel(), Y.ravel()], axis=1)

for ax, A, label in zip(axes, [A1, A2], ["A1 (SPD)", "A2 (not SPD)"]):
    Z = np.einsum('ij,jk,ik->i', pts, A, pts).reshape(X.shape)
    ax.contour(X, Y, Z, levels=[1.0], colors='steelblue', linewidths=2)
    ax.contourf(X, Y, Z, levels=[0, 1.0], colors=['steelblue'], alpha=0.2)
    ax.set_title(f"Unit ball under $\\langle x,y \\rangle = x^\\top {label.split()[0]} y$\n{label}", fontsize=10)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Cauchy-Schwarz Inequality: Numerical Verification

The **Cauchy-Schwarz inequality** states:
$$|\langle \mathbf{x}, \mathbf{y} \rangle| \leq \|\mathbf{x}\| \, \|\mathbf{y}\|$$

This is what makes the angle formula $\cos\omega = \langle \mathbf{x}, \mathbf{y} \rangle / (\|\mathbf{x}\| \|\mathbf{y}\|)$ well-defined: the ratio never exceeds 1 in absolute value.

**Equality** holds if and only if $\mathbf{x}$ and $\mathbf{y}$ are linearly dependent (one is a scalar multiple of the other).

The angle between two vectors is then:
$$\cos\omega = \frac{\langle \mathbf{x}, \mathbf{y} \rangle}{\|\mathbf{x}\| \, \|\mathbf{y}\|}$$

In [ ]:
def cauchy_schwarz_check(x, y):
    """Compute both sides of Cauchy-Schwarz and verify the inequality."""
    lhs = abs(np.dot(x, y))
    rhs = np.linalg.norm(x) * np.linalg.norm(y)
    ratio = lhs / rhs
    cos_angle = np.dot(x, y) / rhs
    angle_deg = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))
    return lhs, rhs, ratio, angle_deg

print("=== Cauchy-Schwarz: |⟨x,y⟩| ≤ ‖x‖·‖y‖ ===")
print(f"{'Pair':<30} {'|⟨x,y⟩|':>10} {'‖x‖·‖y‖':>10} {'ratio':>8} {'angle':>8}")
print("-" * 70)

# Random vector pairs
test_cases = [
    ("Random 3D pair",
     rng.standard_normal(3), rng.standard_normal(3)),
    ("Random 10D pair",
     rng.standard_normal(10), rng.standard_normal(10)),
    ("Parallel (equality)",
     np.array([1.0, 2.0, 3.0]), 2.5 * np.array([1.0, 2.0, 3.0])),
    ("Orthogonal (inner prod=0)",
     np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0])),
    ("Anti-parallel",
     np.array([1.0, 2.0]), -3.0 * np.array([1.0, 2.0])),
]

for label, x, y in test_cases:
    lhs, rhs, ratio, angle = cauchy_schwarz_check(x, y)
    satisfied = "✓" if lhs <= rhs + 1e-10 else "✗"
    print(f"{label:<30} {lhs:>10.4f} {rhs:>10.4f} {ratio:>8.4f} {angle:>7.1f}°  {satisfied}")

# Verify over 10000 random pairs
print("\n--- Batch verification over 10,000 random pairs (5D) ---")
X = rng.standard_normal((10000, 5))
Y = rng.standard_normal((10000, 5))
lhs_batch = np.abs(np.einsum('ij,ij->i', X, Y))
rhs_batch = np.linalg.norm(X, axis=1) * np.linalg.norm(Y, axis=1)
all_satisfied = np.all(lhs_batch <= rhs_batch + 1e-9)
print(f"All 10,000 pairs satisfy |⟨x,y⟩| ≤ ‖x‖·‖y‖: {all_satisfied}")
print(f"Max ratio |⟨x,y⟩| / (‖x‖·‖y‖): {(lhs_batch/rhs_batch).max():.6f}  (should be ≤ 1)")

---
## 4. Orthogonal Projection onto a 1D Subspace

Given a line through the origin spanned by $\mathbf{b}$ and a point $\mathbf{x}$, the **orthogonal projection** of $\mathbf{x}$ onto $\text{span}(\mathbf{b})$ is the closest point in that line:

$$\pi_U(\mathbf{x}) = \frac{\mathbf{b}^\top \mathbf{x}}{\mathbf{b}^\top \mathbf{b}} \, \mathbf{b} = P_\pi \mathbf{x}$$

where the **projection matrix** is:
$$P_\pi = \frac{\mathbf{b}\mathbf{b}^\top}{\mathbf{b}^\top \mathbf{b}}$$

Key properties:
- $P_\pi^2 = P_\pi$ (idempotent — projecting twice gives same result)
- The **residual** $\mathbf{x} - \pi_U(\mathbf{x})$ is orthogonal to $\mathbf{b}$
- This is the right-angle you see in the geometric picture

In [ ]:
def project_1d(x, b):
    """Project vector x onto the 1D subspace spanned by b."""
    b = b / np.linalg.norm(b)  # normalize for clarity (formula works either way)
    lam = np.dot(b, x)         # coordinate
    proj = lam * b             # projection point
    return proj, lam, b

# Define the subspace direction and the vector to project
b = np.array([2.0, 1.0])   # direction of the subspace
x = np.array([1.0, 3.0])   # vector to project

proj, lam, b_hat = project_1d(x, b)
residual = x - proj

print("Projection of x onto span(b):")
print(f"  b (unnormalized) = {b}")
print(f"  b̂ (unit vector)  = {b_hat}")
print(f"  x                = {x}")
print(f"  coordinate λ     = {lam:.4f}")
print(f"  π(x) = λb̂        = {proj}")
print(f"  residual x - π(x)= {residual}")
print(f"  ⟨residual, b̂⟩    = {np.dot(residual, b_hat):.2e}  (should be ≈ 0)")

# Projection matrix P_pi
b_raw = np.array([2.0, 1.0])
P = np.outer(b_raw, b_raw) / np.dot(b_raw, b_raw)
print(f"\nProjection matrix P_π = bb^T / b^Tb:")
print(P)
print(f"Idempotent P²=P: {np.allclose(P @ P, P)}")
print(f"Symmetric P=P^T: {np.allclose(P, P.T)}")

# --- 2D Plot ---
fig, ax = plt.subplots(figsize=(7, 6))

origin = np.zeros(2)
colors = {'x': 'steelblue', 'b': 'darkgreen', 'proj': 'firebrick', 'resid': 'darkorange'}

# The subspace line
t = np.linspace(-0.5, 2.5, 100)
line_pts = np.outer(t, b_hat)
ax.plot(line_pts[:, 0], line_pts[:, 1], 'k--', alpha=0.4, linewidth=1, label='Subspace span(b)')

# Vectors
def draw_arrow(ax, start, end, color, label, lw=2):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    mid = (start + end) / 2
    ax.text(mid[0] + 0.08, mid[1] + 0.08, label, color=color, fontsize=11, fontweight='bold')

draw_arrow(ax, origin, x, colors['x'], '$\\mathbf{x}$')
draw_arrow(ax, origin, proj, colors['proj'], '$\\pi(\\mathbf{x})$')
draw_arrow(ax, proj, x, colors['resid'], '$\\mathbf{x} - \\pi(\\mathbf{x})$')
draw_arrow(ax, origin, b_hat * 1.5, colors['b'], '$\\mathbf{b}$')

# Right-angle marker
sq_size = 0.12
perp_dir = residual / np.linalg.norm(residual) * sq_size
b_dir = b_hat * sq_size
corner = proj + perp_dir
sq_pts = np.array([proj, corner, corner + b_dir, proj + b_dir, proj])
ax.plot(sq_pts[:, 0], sq_pts[:, 1], 'k-', linewidth=1.2)

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3)
ax.set_title("Orthogonal Projection of $\\mathbf{x}$ onto $\\mathrm{span}(\\mathbf{b})$", fontsize=12)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Gram-Schmidt Orthogonalization from Scratch

Given any basis $\{\mathbf{b}_1, \ldots, \mathbf{b}_n\}$, **Gram-Schmidt** constructs an orthonormal basis $\{\mathbf{u}_1, \ldots, \mathbf{u}_n\}$ spanning the same space:

$$\mathbf{u}_1 = \frac{\mathbf{b}_1}{\|\mathbf{b}_1\|}$$

$$\tilde{\mathbf{u}}_k = \mathbf{b}_k - \sum_{i=1}^{k-1} \langle \mathbf{b}_k, \mathbf{u}_i \rangle \, \mathbf{u}_i \qquad \mathbf{u}_k = \frac{\tilde{\mathbf{u}}_k}{\|\tilde{\mathbf{u}}_k\|}$$

Each step subtracts the projection of $\mathbf{b}_k$ onto all previously orthogonalized vectors, leaving only the component orthogonal to them.

The result satisfies:
- $\langle \mathbf{u}_i, \mathbf{u}_j \rangle = 0$ for $i \neq j$ (orthogonal)
- $\|\mathbf{u}_i\| = 1$ for all $i$ (unit length)
- $\text{span}\{\mathbf{u}_1, \ldots, \mathbf{u}_n\} = \text{span}\{\mathbf{b}_1, \ldots, \mathbf{b}_n\}$

In [ ]:
def gram_schmidt(B, verbose=False):
    """
    Gram-Schmidt orthonormalization.
    Input: B — (n, k) matrix whose columns are k linearly independent vectors in R^n.
    Output: U — (n, k) matrix with orthonormal columns.
    """
    n, k = B.shape
    U = np.zeros_like(B, dtype=float)

    for j in range(k):
        v = B[:, j].astype(float)
        if verbose:
            print(f"\nStep {j+1}: start with b{j+1} = {v}")

        # Subtract projections onto all previous orthonormal vectors
        for i in range(j):
            proj_coeff = np.dot(U[:, i], v)
            v = v - proj_coeff * U[:, i]
            if verbose:
                print(f"  Subtract proj onto u{i+1}: coeff={proj_coeff:.4f}, v → {v}")

        # Normalize
        norm_v = np.linalg.norm(v)
        U[:, j] = v / norm_v
        if verbose:
            print(f"  Normalize: ‖v‖={norm_v:.4f}, u{j+1} = {U[:, j]}")

    return U


# Example: 3 linearly independent vectors in R^3
B = np.array([[1.0, 1.0, 0.0],
              [1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0]])

print("Input basis B (columns):")
print(B)
print(f"rank(B) = {np.linalg.matrix_rank(B)}  (columns are linearly independent)")

U = gram_schmidt(B, verbose=True)

print("\n=== Result ===")
print(f"Orthonormal basis U (columns):")
print(U)

# Verify orthonormality: U^T U should be the identity matrix
gram = U.T @ U
print(f"\nVerification — U^T U (should be identity):")
print(gram)
print(f"U^T U ≈ I: {np.allclose(gram, np.eye(U.shape[1]))}")

# Verify same span: every column of B should be expressible in terms of U
# i.e., B should lie in the column space of U → (I - UU^T)B ≈ 0
P_U = U @ U.T  # projection onto span(U)
residual_B = B - P_U @ B
print(f"\n‖B - P_U B‖_F = {np.linalg.norm(residual_B):.2e}  (span preserved: ≈ 0)")

# Cross-check with numpy's QR decomposition
Q, R = np.linalg.qr(B)
print(f"\nNumPy QR gives same span: {np.allclose(np.abs(U.T @ Q), np.eye(3), atol=1e-10)}")
print("(columns may differ by sign — that is fine)")

---
## 6. 2D Rotation Matrix $R(\theta)$: Transforming a Vector at Multiple Angles

The **2D rotation matrix** rotates vectors counterclockwise by angle $\theta$:

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Its columns are the images of the standard basis vectors $\mathbf{e}_1$ and $\mathbf{e}_2$.

**Properties of rotation matrices:**
- $R(\theta)$ is orthogonal: $R(\theta)^\top = R(\theta)^{-1} = R(-\theta)$
- Distances and angles are preserved: $\|R(\theta)\mathbf{x}\| = \|\mathbf{x}\|$
- $R(\alpha)R(\beta) = R(\alpha + \beta)$ — rotations in 2D commute
- $\det(R(\theta)) = 1$ — area is preserved, orientation is not flipped

In [ ]:
def rotation_matrix_2d(theta):
    """2D rotation matrix for angle theta (radians)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s],
                     [s,  c]])

# Verify properties
theta = np.pi / 6  # 30 degrees
R = rotation_matrix_2d(theta)
print(f"R(π/6) =\n{R}\n")
print(f"R^T R = I (orthogonal): {np.allclose(R.T @ R, np.eye(2))}")
print(f"det(R) = {np.linalg.det(R):.4f}  (should be 1)")
print(f"R(α)R(β) = R(α+β): {np.allclose(rotation_matrix_2d(0.3) @ rotation_matrix_2d(0.5), rotation_matrix_2d(0.8))}")

# Starting vector
v0 = np.array([2.0, 0.5])

# Five angles to visualize
angles = [0, np.pi/6, np.pi/3, np.pi/2, 2*np.pi/3]
angle_labels = ['0°', '30°', '60°', '90°', '120°']
cmap = plt.cm.plasma
colors = [cmap(i / (len(angles) - 1)) for i in range(len(angles))]

fig, ax = plt.subplots(figsize=(8, 7))

# Draw unit circle for reference
th = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(th) * np.linalg.norm(v0), np.sin(th) * np.linalg.norm(v0),
        'k--', alpha=0.2, linewidth=1, label=f'Circle r={np.linalg.norm(v0):.2f}')

# Rotated vectors
for angle, label, color in zip(angles, angle_labels, colors):
    R_t = rotation_matrix_2d(angle)
    v_rot = R_t @ v0

    ax.annotate('', xy=v_rot, xytext=np.zeros(2),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.2))
    ax.plot(*v_rot, 'o', color=color, markersize=7)
    # Label slightly outside the tip
    offset = v_rot / np.linalg.norm(v_rot) * 0.25
    ax.text(v_rot[0] + offset[0], v_rot[1] + offset[1], label,
            color=color, fontsize=11, ha='center', fontweight='bold')

    # Verify length preservation
    assert np.isclose(np.linalg.norm(v_rot), np.linalg.norm(v0)), "Length changed!"

ax.set_xlim(-3, 3)
ax.set_ylim(-2, 3)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3)
ax.set_title(f"2D Rotation $R(\\theta)$ applied to $\\mathbf{{v}}_0 = {v0}$ at 5 angles\n"
             f"($\\|v_0\\| = {np.linalg.norm(v0):.3f}$ preserved throughout)", fontsize=11)
ax.legend(fontsize=9)

# Show the rotation matrix for each angle in a table below
print(f"\nOriginal vector v0 = {v0}, ‖v0‖ = {np.linalg.norm(v0):.4f}")
print(f"\n{'Angle':<8} {'R(θ)v0':>20} {'‖R(θ)v0‖':>12}")
print("-" * 45)
for angle, label in zip(angles, angle_labels):
    v_rot = rotation_matrix_2d(angle) @ v0
    print(f"{label:<8} [{v_rot[0]:>7.4f}, {v_rot[1]:>7.4f}]  {np.linalg.norm(v_rot):>10.4f}")

plt.tight_layout()
plt.show()

---
## Exercises

**Exercise 1 — Lp norm unit balls:**
For $p \in \{0.5, 1, 1.5, 2, 3, \infty\}$, plot the unit ball $\{\mathbf{x} \in \mathbb{R}^2 : \|\mathbf{x}\|_p = 1\}$ in a single figure. Use `np.linalg.norm(v, ord=p)` and sample many candidate points, keeping only those with norm $\approx 1$. Observe how the shape transitions from non-convex ($p < 1$) to diamond ($p=1$) to circle ($p=2$) to square ($p=\infty$).

**Exercise 2 — Projection onto a 2D subspace:**
Let $U$ be spanned by $\mathbf{b}_1 = [1, 0, 1]^\top$ and $\mathbf{b}_2 = [0, 1, 1]^\top$ in $\mathbb{R}^3$. Project $\mathbf{x} = [1, 2, 3]^\top$ onto $U$ using the formula $\pi_U(\mathbf{x}) = B(B^\top B)^{-1} B^\top \mathbf{x}$ where $B = [\mathbf{b}_1 \mid \mathbf{b}_2]$. Verify that the residual $\mathbf{x} - \pi_U(\mathbf{x})$ is orthogonal to both $\mathbf{b}_1$ and $\mathbf{b}_2$.

**Exercise 3 — Gram-Schmidt on random vectors:**
Generate 4 random vectors in $\mathbb{R}^5$ using `rng.standard_normal((5, 4))`. Apply `gram_schmidt` from above. Verify that the resulting $5 \times 4$ matrix $U$ satisfies $U^\top U = I_4$. Compare with `np.linalg.qr`.

**Exercise 4 — 3D rotation and non-commutativity:**
Implement 3D rotation matrices $R_x(\theta)$ and $R_z(\theta)$ (rotation about the $x$-axis and $z$-axis respectively, as given in MML §3.9). Pick $\theta = \pi/4$ and a vector $\mathbf{v} = [1, 0, 0]^\top$. Compute $R_x R_z \mathbf{v}$ and $R_z R_x \mathbf{v}$. Are they equal? Verify that $\det(R) = 1$ and $R^\top R = I$ for both.